# Batch Beam Profiler MFD Analysis

Edit `BASE_DIR` and `MAG` in the code cell, then run it. Any **`.txt` file whose filename contains `data`** (case-insensitive) is processed, regardless of the rest of its name. Each gets a same-name output subfolder.


In [3]:
# -*- coding: utf-8 -*-
"""
Batch Thorlabs beam-profile MFD analysis

What this script does
---------------------
1. Scans BASE_DIR for .txt files whose filename contains DATA_KEYWORD ("data" by default).
2. Processes each matching file independently.
3. Creates a sub-directory beside the input file named exactly like the input filename
   without its extension.
4. Saves all per-file outputs inside that sub-directory.

Supported input layouts
-----------------------
A) 2-column profile export:
      position    intensity
   The axis is inferred from the filename:
      X-data / X_Data / X data -> X rule
      Y-data / Y_Data / Y data -> Y rule

B) 4-column X-Y profile export:
      X_position    X_intensity    Y_position    Y_intensity
   Both profiles are processed in one pass.

MFD rules preserved from the original notebook
-----------------------------------------------
- X MFD: outermost 13.5% (~1/e^2) crossings across the whole X trace.
- Y MFD: immediate left/right 13.5% crossings around the highest Y peak.

Outputs inside each file-specific subfolder
-------------------------------------------
- <input_stem>_profile.pdf
- <input_stem>_profile.png
- <input_stem>_profile.jpg          (optional)
- analysis_results.txt
- normalized_profile_X.csv          (when X exists)
- normalized_profile_Y.csv          (when Y exists)

Optional:
- batch_MFD_summary.csv in BASE_DIR, collecting all results.

Place this notebook in the intended data folder and open/run it from that folder.
The working directory is detected automatically; adjust MAG if needed.
"""

import csv
import re
import traceback
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# =============================================================================
# USER SETTINGS
# =============================================================================

# Analyze the current Jupyter working directory (no hard-coded C:/ or E:/ paths).
# Open the notebook from the folder containing the input .txt files.
BASE_DIR = Path.cwd().resolve()

# Match filenames containing this text, case-insensitively.
DATA_KEYWORD = "data"

# The scan is intentionally NON-RECURSIVE: only files directly inside BASE_DIR
# are processed. This avoids re-processing output files inside the new folders.
SCAN_RECURSIVE = False

# Only .txt files are eligible, regardless of filename formatting or capitalization.
ALLOWED_EXTENSIONS = {".txt"}

# -------------------------------------------------------------------------
# MAGNIFICATION
# -------------------------------------------------------------------------

MAG = 7.0       # 97 mm link
# MAG = 8.95    # 47 mm link

# -------------------------------------------------------------------------
# DATA PROCESSING
# -------------------------------------------------------------------------

EDGE_FRAC = 0.20
LEVEL = 13.5    # percent, approximately 1/e^2

# -------------------------------------------------------------------------
# FIGURE SETTINGS
# -------------------------------------------------------------------------

FIG_WIDTH_IN = 8.0
FIG_HEIGHT_IN = 6.0
PNG_DPI = 600

# -------------------------------------------------------------------------
# SAVE SETTINGS
# -------------------------------------------------------------------------

SAVE_PDF = True
SAVE_PNG = True
SAVE_JPG = False
JPG_DPI = 600
JPG_QUALITY = 95
SAVE_TRANSPARENT = False

# Save an aggregate CSV in BASE_DIR after all files are processed.
SAVE_MASTER_SUMMARY = True

# Normally False for unattended batch processing.
SHOW_FIGURES = False


# =============================================================================
# APPROVED JOURNAL PLOT STYLE
# =============================================================================

def apply_style():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "mathtext.fontset": "stix",
        "text.usetex": False,
        "axes.labelsize": 36,
        "axes.titlesize": 32,
        "xtick.labelsize": 25,
        "ytick.labelsize": 25,
        "legend.fontsize": 24,
        "axes.linewidth": 2.0,
        "xtick.major.width": 2.0,
        "ytick.major.width": 2.0,
        "xtick.major.size": 8,
        "ytick.major.size": 8,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.minor.visible": False,
        "ytick.minor.visible": False,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


apply_style()


# =============================================================================
# HELPERS
# =============================================================================

def safe_float(value):
    """Convert one field to float; return NaN if conversion fails."""
    try:
        return float(str(value).strip())
    except Exception:
        return np.nan


def find_matching_files(base_dir):
    """Select only .txt files with DATA_KEYWORD anywhere in the filename (case-insensitive)."""
    iterator = base_dir.rglob("*") if SCAN_RECURSIVE else base_dir.glob("*")

    files = []
    for path in iterator:
        if not path.is_file():
            continue
        if DATA_KEYWORD.lower() not in path.name.lower():
            continue
        if ALLOWED_EXTENSIONS is not None and path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue
        files.append(path)

    return sorted(files, key=lambda p: p.name.lower())


def read_numeric_export(path):
    """
    Read a tab/comma/whitespace-like text export.

    Returns
    -------
    dict with:
        layout: "2col" or "4col"
        x: (position, intensity) or None
        y: (position, intensity) or None
        generic: (position, intensity) or None

    For 4-column data, columns are assumed to be:
        X_position, X_intensity, Y_position, Y_intensity.
    """
    with open(path, "r", encoding="latin-1", newline="") as f:
        raw_lines = f.readlines()

    parsed_rows = []

    for line in raw_lines:
        line = line.strip("\r\n")
        if not line.strip():
            continue

        # Prefer tabs. Fall back to comma, semicolon, then generic whitespace.
        if "\t" in line:
            fields = line.split("\t")
        elif "," in line:
            fields = line.split(",")
        elif ";" in line:
            fields = line.split(";")
        else:
            fields = re.split(r"\s+", line.strip())

        vals = [safe_float(v) for v in fields[:4]]

        # A numeric data row needs at least the first two numeric values.
        if len(vals) >= 2 and np.isfinite(vals[0]) and np.isfinite(vals[1]):
            while len(vals) < 4:
                vals.append(np.nan)
            parsed_rows.append(vals[:4])

    if not parsed_rows:
        raise ValueError("Could not locate numeric profile data in the file.")

    arr = np.asarray(parsed_rows, dtype=float)

    # Determine whether valid columns 3 and 4 exist often enough to be a real Y trace.
    have_y_pair = (
        arr.shape[1] >= 4
        and np.count_nonzero(np.isfinite(arr[:, 2]) & np.isfinite(arr[:, 3])) >= 3
    )

    if have_y_pair:
        xm = np.isfinite(arr[:, 0]) & np.isfinite(arr[:, 1])
        ym = np.isfinite(arr[:, 2]) & np.isfinite(arr[:, 3])

        return {
            "layout": "4col",
            "x": (arr[xm, 0], arr[xm, 1]),
            "y": (arr[ym, 2], arr[ym, 3]),
            "generic": None,
        }

    m = np.isfinite(arr[:, 0]) & np.isfinite(arr[:, 1])

    return {
        "layout": "2col",
        "x": None,
        "y": None,
        "generic": (arr[m, 0], arr[m, 1]),
    }


def infer_axis_from_filename(path):
    """
    Infer X or Y from common filename patterns.
    Examples:
        100um_WG #1_X-data.txt
        100um_WG #1_Y-data.txt
        W5+10_X_Data.txt
    """
    stem = path.stem.lower()

    x_patterns = [
        r"(^|[^a-z])x[\s_-]*data([^a-z]|$)",
        r"data[\s_-]*x([^a-z]|$)",
    ]
    y_patterns = [
        r"(^|[^a-z])y[\s_-]*data([^a-z]|$)",
        r"data[\s_-]*y([^a-z]|$)",
    ]

    if any(re.search(p, stem) for p in x_patterns):
        return "X"
    if any(re.search(p, stem) for p in y_patterns):
        return "Y"

    if "_x-" in stem or "_x_" in stem or stem.endswith("_x"):
        return "X"
    if "_y-" in stem or "_y_" in stem or stem.endswith("_y"):
        return "Y"

    return "UNKNOWN"


# =============================================================================
# BASELINE CORRECTION AND NORMALIZATION
# =============================================================================

def baseline_and_normalize(pos, val, edge_frac=0.20):
    pos = np.asarray(pos, dtype=float)
    val = np.asarray(val, dtype=float)

    mask = np.isfinite(pos) & np.isfinite(val)
    pos = pos[mask]
    val = val[mask]

    if len(pos) < 3:
        raise ValueError("Fewer than 3 valid profile samples remain after filtering.")

    order = np.argsort(pos)
    pos = pos[order]
    val = val[order]

    n = len(val)
    k = max(1, int(edge_frac * n))

    baseline = np.median(np.r_[val[:k], val[-k:]])

    corrected = np.clip(val - baseline, 0, None)
    peak = np.max(corrected)

    if not np.isfinite(peak) or peak <= 0:
        raise ValueError("Profile peak is zero/non-finite after baseline subtraction.")

    normalized = corrected / peak * 100.0

    return pos, normalized, baseline, peak


# =============================================================================
# X MFD: OUTERMOST CROSSINGS ACROSS THE ENTIRE TRACE
# =============================================================================

def width_at_level(pos, val, level=13.5):
    if len(pos) < 3:
        return np.nan, np.nan, np.nan

    order = np.argsort(pos)
    x = np.asarray(pos)[order]
    y = np.asarray(val)[order]

    crossings = []

    for i in range(len(x) - 1):
        y0 = y[i]
        y1 = y[i + 1]

        if not (np.isfinite(y0) and np.isfinite(y1)):
            continue

        if (y0 - level) * (y1 - level) <= 0 and y1 != y0:
            t = (level - y0) / (y1 - y0)
            xc = x[i] + t * (x[i + 1] - x[i])
            crossings.append(float(xc))

    if len(crossings) >= 2:
        crossings.sort()
        return crossings[-1] - crossings[0], crossings[0], crossings[-1]

    return np.nan, np.nan, np.nan


# =============================================================================
# Y MFD: IMMEDIATE CROSSINGS AROUND THE HIGHEST PEAK
# =============================================================================

def width_at_level_highest_peak(pos, val, level=13.5):
    if len(pos) < 3:
        return np.nan, np.nan, np.nan

    order = np.argsort(pos)
    x = np.asarray(pos)[order]
    y = np.asarray(val)[order]

    i_peak = int(np.nanargmax(y))

    x_left = np.nan
    for i in range(i_peak - 1, -1, -1):
        y0 = y[i]
        y1 = y[i + 1]

        if not (np.isfinite(y0) and np.isfinite(y1)):
            continue

        if (y0 - level) * (y1 - level) <= 0 and y1 != y0:
            t = (level - y0) / (y1 - y0)
            x_left = x[i] + t * (x[i + 1] - x[i])
            break

    x_right = np.nan
    for i in range(i_peak, len(x) - 1):
        y0 = y[i]
        y1 = y[i + 1]

        if not (np.isfinite(y0) and np.isfinite(y1)):
            continue

        if (y0 - level) * (y1 - level) <= 0 and y1 != y0:
            t = (level - y0) / (y1 - y0)
            x_right = x[i] + t * (x[i + 1] - x[i])
            break

    if np.isfinite(x_left) and np.isfinite(x_right) and x_right >= x_left:
        return x_right - x_left, x_left, x_right

    return np.nan, np.nan, np.nan


# =============================================================================
# SAVE NORMALIZED DATA
# =============================================================================

def save_profile_csv(path, pos, norm):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["position_um", "normalized_intensity_percent"])
        for p, v in zip(pos, norm):
            writer.writerow([f"{p:.9g}", f"{v:.9g}"])


# =============================================================================
# PLOTTING
# =============================================================================

def format_axes(ax, title, xmin, xmax):
    ax.axhline(
        LEVEL,
        linestyle="--",
        linewidth=3.0,
        color="black",
        alpha=0.8,
    )

    ax.text(
        0.02,
        LEVEL + 3,
        r"$13.5\% \; (\approx 1/e^2)$",
        fontsize=24,
        ha="left",
        va="bottom",
        transform=ax.get_yaxis_transform(),
    )

    ax.set_xlabel("Position (µm)")
    ax.set_ylabel("Normalized intensity (%)")
    ax.set_title(title, pad=12)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-2, 105)

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        top=True,
        right=True,
        width=2.0,
        length=8,
        labelsize=25,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(2.0)

    ax.grid(True, which="major", linewidth=0.6, alpha=0.20)
    ax.legend(loc="upper right", frameon=False, fontsize=24)


def save_figure(fig, out_dir, basename):
    if SAVE_PDF:
        fig.savefig(
            out_dir / f"{basename}.pdf",
            bbox_inches="tight",
            transparent=SAVE_TRANSPARENT,
        )

    if SAVE_PNG:
        fig.savefig(
            out_dir / f"{basename}.png",
            dpi=PNG_DPI,
            bbox_inches="tight",
            transparent=SAVE_TRANSPARENT,
        )

    if SAVE_JPG:
        fig.savefig(
            out_dir / f"{basename}.jpg",
            dpi=JPG_DPI,
            bbox_inches="tight",
            pil_kwargs={"quality": JPG_QUALITY},
        )


# =============================================================================
# PROCESS ONE FILE
# =============================================================================

def process_one_file(file_path):
    print("\n" + "=" * 90)
    print(f"PROCESSING: {file_path.name}")
    print("=" * 90)

    # Folder has exactly the same base filename as the source file.
    out_dir = file_path.parent / file_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    export = read_numeric_export(file_path)

    result = {
        "input_file": file_path.name,
        "layout": export["layout"],
        "axis": "",
        "mfd_x_um": np.nan,
        "mfd_y_um": np.nan,
        "x_left_um": np.nan,
        "x_right_um": np.nan,
        "y_left_um": np.nan,
        "y_right_um": np.nan,
        "status": "OK",
        "output_directory": str(out_dir),
    }

    # ---------------------------------------------------------------------
    # 4-COLUMN COMBINED X-Y FILE
    # ---------------------------------------------------------------------
    if export["layout"] == "4col":
        result["axis"] = "X+Y"

        x_pos_raw, x_raw = export["x"]
        y_pos_raw, y_raw = export["y"]

        x_pos_raw = x_pos_raw / MAG
        y_pos_raw = y_pos_raw / MAG

        x_pos, x_val, x_base, x_peak = baseline_and_normalize(
            x_pos_raw, x_raw, EDGE_FRAC
        )
        y_pos, y_val, y_base, y_peak = baseline_and_normalize(
            y_pos_raw, y_raw, EDGE_FRAC
        )

        x_w, x_l, x_r = width_at_level(x_pos, x_val, LEVEL)
        y_w, y_l, y_r = width_at_level_highest_peak(y_pos, y_val, LEVEL)

        result.update({
            "mfd_x_um": x_w,
            "mfd_y_um": y_w,
            "x_left_um": x_l,
            "x_right_um": x_r,
            "y_left_um": y_l,
            "y_right_um": y_r,
        })

        save_profile_csv(out_dir / "normalized_profile_X.csv", x_pos, x_val)
        save_profile_csv(out_dir / "normalized_profile_Y.csv", y_pos, y_val)

        fig, ax = plt.subplots(
            figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
            dpi=PNG_DPI,
        )

        ax.plot(x_pos, x_val, linewidth=4.0, label="X profile")
        ax.plot(y_pos, y_val, linewidth=4.0, label="Y profile")

        mfd_text = f"MFDx = {x_w:.2f} µm\nMFDy = {y_w:.2f} µm"
        ax.text(
            0.03,
            0.96,
            mfd_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=24,
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor="white",
                edgecolor="gray",
                linewidth=1.5,
                alpha=0.90,
            ),
        )

        xmin = min(np.min(x_pos), np.min(y_pos))
        xmax = max(np.max(x_pos), np.max(y_pos))
        format_axes(ax, file_path.stem, xmin, xmax)
        fig.tight_layout()

        save_figure(fig, out_dir, f"{file_path.stem}_profile")

        results_text = f"""THORLABS X-Y PROFILE MFD ANALYSIS
Input file: {file_path.name}
Input layout: 4-column combined X-Y export
Magnification: {MAG}x
1/e^2 level: {LEVEL} %

X method: outermost crossings over the whole trace
MFD_x: {x_w:.6f} µm
X left crossing: {x_l:.6f} µm
X right crossing: {x_r:.6f} µm
X raw baseline: {x_base:.9g}
X baseline-subtracted peak: {x_peak:.9g}

Y method: nearest crossings around the highest peak
MFD_y: {y_w:.6f} µm
Y left crossing: {y_l:.6f} µm
Y right crossing: {y_r:.6f} µm
Y raw baseline: {y_base:.9g}
Y baseline-subtracted peak: {y_peak:.9g}
"""

    # ---------------------------------------------------------------------
    # 2-COLUMN SINGLE PROFILE FILE
    # ---------------------------------------------------------------------
    else:
        axis = infer_axis_from_filename(file_path)
        result["axis"] = axis

        pos_raw, raw = export["generic"]
        pos_raw = pos_raw / MAG

        pos, val, baseline, peak = baseline_and_normalize(
            pos_raw, raw, EDGE_FRAC
        )

        if axis == "Y":
            method = "nearest 13.5% crossings around the highest peak"
            width, left, right = width_at_level_highest_peak(pos, val, LEVEL)
            result.update({
                "mfd_y_um": width,
                "y_left_um": left,
                "y_right_um": right,
            })
            profile_label = "Y profile"
            mfd_label = "MFDy"
            save_profile_csv(out_dir / "normalized_profile_Y.csv", pos, val)

        else:
            # X is the default if filename is ambiguous.
            # This preserves the original outermost-crossing behavior rather
            # than silently applying the Y/highest-peak rule.
            method = "outermost 13.5% crossings over the whole trace"
            width, left, right = width_at_level(pos, val, LEVEL)

            if axis == "UNKNOWN":
                result["axis"] = "UNKNOWN->X_RULE"

            result.update({
                "mfd_x_um": width,
                "x_left_um": left,
                "x_right_um": right,
            })
            profile_label = "X profile" if axis == "X" else "Profile"
            mfd_label = "MFDx" if axis == "X" else "MFD"
            save_profile_csv(out_dir / "normalized_profile_X.csv", pos, val)

        fig, ax = plt.subplots(
            figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
            dpi=PNG_DPI,
        )

        ax.plot(pos, val, linewidth=4.0, label=profile_label)

        ax.text(
            0.03,
            0.96,
            f"{mfd_label} = {width:.2f} µm",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=24,
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor="white",
                edgecolor="gray",
                linewidth=1.5,
                alpha=0.90,
            ),
        )

        format_axes(ax, file_path.stem, np.min(pos), np.max(pos))
        fig.tight_layout()

        save_figure(fig, out_dir, f"{file_path.stem}_profile")

        results_text = f"""THORLABS SINGLE-PROFILE MFD ANALYSIS
Input file: {file_path.name}
Input layout: 2-column profile export
Detected axis: {result["axis"]}
Magnification: {MAG}x
1/e^2 level: {LEVEL} %

MFD method: {method}
MFD: {width:.6f} µm
Left crossing: {left:.6f} µm
Right crossing: {right:.6f} µm
Raw baseline: {baseline:.9g}
Baseline-subtracted peak: {peak:.9g}
"""

    with open(out_dir / "analysis_results.txt", "w", encoding="utf-8") as f:
        f.write(results_text)

    if SHOW_FIGURES:
        plt.show()

    plt.close(fig)

    print(f"Saved to: {out_dir}")
    if np.isfinite(result["mfd_x_um"]):
        print(f"MFD_x = {result['mfd_x_um']:.3f} µm")
    if np.isfinite(result["mfd_y_um"]):
        print(f"MFD_y = {result['mfd_y_um']:.3f} µm")

    return result


# =============================================================================
# MASTER SUMMARY
# =============================================================================

def write_master_summary(path, rows):
    fields = [
        "input_file",
        "layout",
        "axis",
        "mfd_x_um",
        "mfd_y_um",
        "x_left_um",
        "x_right_um",
        "y_left_um",
        "y_right_um",
        "status",
        "output_directory",
    ]

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


# =============================================================================
# MAIN BATCH WORKFLOW
# =============================================================================

def main():
    if not BASE_DIR.exists():
        raise FileNotFoundError(
            f"BASE_DIR does not exist:\n{BASE_DIR}\n\n"
            "Check the current Jupyter working directory before running."
        )

    matching_files = find_matching_files(BASE_DIR)

    print("=" * 90)
    print("BATCH THORLABS BEAM-PROFILE ANALYSIS")
    print("=" * 90)
    print(f"Folder: {BASE_DIR}")
    print(f"Keyword: {DATA_KEYWORD!r}")
    print(f"Magnification: {MAG}x")
    print(f"Matching files found: {len(matching_files)}")

    if not matching_files:
        print("\nNo matching files were found.")
        return

    results = []

    for file_path in matching_files:
        try:
            result = process_one_file(file_path)
        except Exception as exc:
            print(f"\nFAILED: {file_path.name}")
            print(f"Reason: {exc}")

            # Keep the batch running even if one file fails.
            out_dir = file_path.parent / file_path.stem
            out_dir.mkdir(parents=True, exist_ok=True)

            with open(out_dir / "ERROR.txt", "w", encoding="utf-8") as f:
                f.write(f"Input file: {file_path}\n")
                f.write(f"Error: {exc}\n\n")
                f.write(traceback.format_exc())

            result = {
                "input_file": file_path.name,
                "layout": "",
                "axis": "",
                "mfd_x_um": np.nan,
                "mfd_y_um": np.nan,
                "x_left_um": np.nan,
                "x_right_um": np.nan,
                "y_left_um": np.nan,
                "y_right_um": np.nan,
                "status": f"FAILED: {exc}",
                "output_directory": str(out_dir),
            }

        results.append(result)

    if SAVE_MASTER_SUMMARY:
        summary_path = BASE_DIR / "batch_MFD_summary.csv"
        write_master_summary(summary_path, results)
        print(f"\nMaster summary: {summary_path}")

    ok_count = sum(r["status"] == "OK" for r in results)
    fail_count = len(results) - ok_count

    print("\n" + "=" * 90)
    print("BATCH COMPLETE")
    print("=" * 90)
    print(f"Successful: {ok_count}")
    print(f"Failed:     {fail_count}")
    print(f"Total:      {len(results)}")


if __name__ == "__main__":
    main()


BATCH THORLABS BEAM-PROFILE ANALYSIS
Folder: E:\OneDrive - The Pennsylvania State University\Desktop\BeamProfiler_Data\2027_ECTC\Ref_sample_test\12uJ+10mm.s
Keyword: 'data'
Magnification: 7.0x
Matching files found: 18

PROCESSING: 100um_WG#1_X-data.txt
Saved to: E:\OneDrive - The Pennsylvania State University\Desktop\BeamProfiler_Data\2027_ECTC\Ref_sample_test\12uJ+10mm.s\100um_WG#1_X-data
MFD_x = 10.901 µm
MFD_y = 11.856 µm

PROCESSING: 100um_WG#1_Y-data.txt
Saved to: E:\OneDrive - The Pennsylvania State University\Desktop\BeamProfiler_Data\2027_ECTC\Ref_sample_test\12uJ+10mm.s\100um_WG#1_Y-data
MFD_x = 10.156 µm
MFD_y = 11.816 µm

PROCESSING: 100um_WG#2_X-data.txt
Saved to: E:\OneDrive - The Pennsylvania State University\Desktop\BeamProfiler_Data\2027_ECTC\Ref_sample_test\12uJ+10mm.s\100um_WG#2_X-data
MFD_x = 10.173 µm
MFD_y = 13.513 µm

PROCESSING: 100um_WG#2_Y-data.txt
Saved to: E:\OneDrive - The Pennsylvania State University\Desktop\BeamProfiler_Data\2027_ECTC\Ref_sample_test\12uJ